In [20]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os


In [26]:
load_dotenv()

True

In [34]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # fast + cheap
    temperature=0.7
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [35]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [30]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [36]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [37]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The distance between the Earth and the Moon is not constant because the Moon's orbit around Earth is elliptical, not a perfect circle.

Here are the key distances:

*   **Average Distance:** Approximately **384,400 kilometers (238,900 miles)**
*   **Perigee (closest point):** Approximately **363,104 kilometers (225,623 miles)**
*   **Apogee (farthest point):** Approximately **405,696 kilometers (252,088 miles)**

For most general purposes, the average distance of **384,400 km (238,900 miles)** is the figure commonly cited.

To put it in perspective, you could fit about 30 Earths side-by-side in the space between the Earth and the Moon! Light from the Moon takes about 1.3 seconds to reach Earth.


In [38]:
model.invoke('How far is moon from the earth?').content

'The distance between the Earth and the Moon is not constant, as the Moon orbits the Earth in an elliptical path.\n\nHowever, the **average distance** is approximately:\n\n*   **384,400 kilometers (km)**\n*   **238,900 miles**\n\nThis distance can vary:\n\n*   **At its closest (perigee):** about 363,104 km (225,623 miles)\n*   **At its farthest (apogee):** about 405,696 km (252,088 miles)\n\nTo give you a sense of scale, it takes light about 1.28 seconds to travel from the Moon to the Earth.'